# Exercise 4: Cross-lingual parser evaluation

*Syntax and Universal Dependencies · version 0.1.0*

> Take the Kyrgyz TueCL and Kazakh KTB test sets. Apply the Turkish parser to each and compute UAS and LAS. Now transliterate the test sentences to Latin script (for Kazakh the 2021 Latin alphabet or CTS; for Kyrgyz CTS) and re-evaluate. Keep an explicit word-by-word mapping from transliterated to original tokens. How much does LAS change? Compare with TurkicNLP's multilingual parser on the original Cyrillic text, and, for Azerbaijani TueCL, with the parsers of the hands-on. Which dependency relations remain most error-prone?

**Setup.** `pip install "turkicnlp==0.5.0" "stanza==1.11.0" scipy matplotlib requests`. Stanza is pinned to 1.11.0 because the worked solutions use its Turkish (IMST), Kazakh (KTB) and Kyrgyz (KTMU) checkpoints; with another version, Stanza downloads that version's models and the numbers can change. Models already on disk are used from `STANZA_RESOURCES_DIR` (default `~/stanza_resources`); missing ones are downloaded. The Universal Dependencies files are the **UD 2.17** release files (tag `r2.17` on GitHub). They are downloaded once into `UD217_DIR` (default `~/.cache/ud217`) and checked against the SHA-256 hashes recorded in the book's treebank inventory. Parsers receive the **gold words** (`tokenize_pretokenized=True`), so tokenisation errors do not enter the scores.

**Design.** Each test sentence is parsed from its gold words. Transliteration is applied word by word, so the mapping from converted to original words is one-to-one by construction, and the original word forms are restored before scoring with the official CoNLL 2018 scorer (`stanza.utils.conll18_ud_eval`). Because the input words are gold, `Words` is always 100 and UAS/LAS measure parsing only. References: the Turkish model on Turkish TueCL (the same sentences as the Kyrgyz and Azerbaijani sets), the Kyrgyz KTMU model on Kyrgyz TueCL (an in-language model trained on another treebank), and the Kazakh KTB model on KTB, which is **not** held out: Stanza's preparation of KTB swaps its small training split with the test split, so that row is a replay diagnostic (Syntax and Universal Dependencies). TurkicNLP's multilingual Glot500 parser and its custom Azerbaijani Stanza model were not available in the environment used for the worked solution.

In [ ]:
import collections, hashlib, math, os, random, statistics
from pathlib import Path
import requests

# Pinned UD 2.17 release files (the inventory in tables/tbl-ud217-inventory) and their SHA-256 hashes.
UD = "https://raw.githubusercontent.com/UniversalDependencies/{repo}/r2.17/{file}"
SHA256 = {
    "UD_Turkish-BOUN/tr_boun-ud-train.conllu": "373fe39e3acdba3219521f54fa5ae768851fbb4bf9a31079ad1ca6590410650c",
    "UD_Turkish-BOUN/tr_boun-ud-test.conllu": "4f74fc0f9efdd1285d781f3efd1e98cbf39cf4433df9eb4dc14162e8ed36773b",
    "UD_Turkish-IMST/tr_imst-ud-train.conllu": "8d8b8b581dab9362e6f2e4328aef77b9fabec1d189a199e320ba136710c7bd83",
    "UD_Turkish-IMST/tr_imst-ud-test.conllu": "fc9269bd598489b942a025b678779557266ce0c054a843e47c7db386d76d6459",
    "UD_Turkish-TueCL/tr_tuecl-ud-test.conllu": "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    "UD_Azerbaijani-TueCL/az_tuecl-ud-test.conllu": "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    "UD_Kyrgyz-TueCL/ky_tuecl-ud-test.conllu": "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
    "UD_Kazakh-KTB/kk_ktb-ud-test.conllu": "1100846ee50c2f970562a21dca8415a381c470ea06bdb8496ae213d59c3a3c4e",
    "UD_English-EWT/en_ewt-ud-train.conllu": "6fe5692f2e02198262483daba0626efe346dd19f0ce6c3ee7728b08af9b90a9c",
}
CACHE = Path(os.environ.get("UD217_DIR", "~/.cache/ud217")).expanduser()

def ud_path(key):
    """Local copy of a pinned UD 2.17 file, downloaded once and checked against its SHA-256 hash."""
    path = CACHE / key
    if not path.exists():
        repo, file = key.split("/")
        r = requests.get(UD.format(repo=repo, file=file), timeout=120)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    if hashlib.sha256(path.read_bytes()).hexdigest() != SHA256[key]:
        raise ValueError(f"{key} differs from the UD 2.17 release file")
    return path

Word = collections.namedtuple("Word", "id form lemma upos xpos feats head deprel")

def read(key_or_path):
    """Sentences as dicts: sent_id, text and words (integer-ID syntactic words only;
    multiword-token ranges and empty nodes are skipped)."""
    path = ud_path(key_or_path) if key_or_path in SHA256 else Path(key_or_path)
    sents, meta, words = [], {}, []
    for line in path.read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words:
                sents.append({"sent_id": meta.get("sent_id"), "text": meta.get("text"), "words": words})
            meta, words = {}, []
        elif line.startswith("#"):
            k, _, v = line[1:].partition("=")
            meta.setdefault(k.strip(), v.strip())
        else:
            c = line.split("\t")
            if c[0].isdigit():
                words.append(Word(int(c[0]), c[1], c[2], c[3], c[4], c[5], int(c[6]), c[7]))
    return sents

def feats(w):
    return dict(f.split("=", 1) for f in w.feats.split("|") if "=" in f)

def children(words):
    out = collections.defaultdict(list)
    for w in words:
        out[w.head].append(w)
    return out

def subtree(words, i, kids=None):
    """IDs of word i and all its descendants."""
    kids = kids or children(words)
    out, stack = set(), [i]
    while stack:
        j = stack.pop(); out.add(j); stack.extend(k.id for k in kids[j])
    return out

def q(values, p):
    """Nearest-rank quantile, as in the chapter's arc-length table."""
    v = sorted(values)
    return v[math.ceil(len(v) * p) - 1] if v else None

def stanza_pipeline(lang, processors="tokenize,pos,lemma,depparse"):
    """A Stanza pipeline over gold words (tokenize_pretokenized=True). Uses models already on disk
    (STANZA_RESOURCES_DIR or ~/stanza_resources) and downloads them only if they are missing."""
    import logging, stanza
    logging.getLogger("stanza").setLevel(logging.ERROR)
    model_dir = os.environ.get("STANZA_RESOURCES_DIR", str(Path("~/stanza_resources").expanduser()))
    have = (Path(model_dir) / lang / "depparse").exists()
    return stanza.Pipeline(lang, processors=processors, tokenize_pretokenized=True, model_dir=model_dir,
                           download_method=None if have else "reuse_resources", verbose=False)

def parse_gold_words(nlp, sents, forms=None):
    """Parse each gold sentence from its gold words (optionally replaced by `forms`, one list per
    sentence, e.g. transliterated) and return predicted sentences with the original word forms."""
    forms = forms or [[w.form for w in s["words"]] for s in sents]
    doc = nlp(forms)
    out = []
    for s, ps in zip(sents, doc.sentences):
        assert len(ps.words) == len(s["words"]), s["sent_id"]
        out.append({"sent_id": s["sent_id"], "text": s["text"], "words": [
            Word(g.id, g.form, p.lemma or "_", p.upos or "_", p.xpos or "_", p.feats or "_", p.head, p.deprel)
            for g, p in zip(s["words"], ps.words)]})
    return out

def write_conllu(sents, path):
    with open(path, "w", encoding="utf-8") as f:
        for s in sents:
            f.write(f"# sent_id = {s['sent_id']}\n# text = {s['text']}\n")
            for w in s["words"]:
                f.write("\t".join([str(w.id), w.form, w.lemma or "_", w.upos, w.xpos or "_", w.feats or "_",
                                   str(w.head), w.deprel, "_", "_"]) + "\n")
            f.write("\n")

def score(gold, pred):
    """UPOS, UAS and LAS (%) with the official CoNLL 2018 scorer, over gold-word input.
    Gold multiword tokens are not written, so both files have the same words and text."""
    import tempfile
    from stanza.utils import conll18_ud_eval as ev
    with tempfile.TemporaryDirectory() as tmp:
        g, p = Path(tmp) / "gold.conllu", Path(tmp) / "pred.conllu"
        write_conllu(gold, g); write_conllu(pred, p)
        r = ev.evaluate(ev.load_conllu_file(str(g)), ev.load_conllu_file(str(p)))
    return {k: round(100 * r[k].f1, 2) for k in ("Words", "UPOS", "UAS", "LAS")}

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

def convert(sents, lang, target):
    """Transliterate word by word, so every converted word stays aligned with its gold word."""
    t = Transliterator(lang, Script("Cyrl"), Script(target))
    return [[t.transliterate(w.form) for w in s["words"]] for s in sents]

def per_relation(gold, pred, top=6):
    """Relations with the lowest labelled recall (head and coarse label right), among relations with >= 20 arcs."""
    n, ok = collections.Counter(), collections.Counter()
    for g, p in zip(gold, pred):
        for gw, pw in zip(g["words"], p["words"]):
            r = gw.deprel.split(":")[0]
            n[r] += 1; ok[r] += pw.head == gw.head and pw.deprel.split(":")[0] == r
    worst = sorted((ok[r] / n[r], r) for r in n if n[r] >= 20)[:top]
    return ", ".join(f"{r} {a:.0%} (n={n[r]})" for a, r in worst)

nlp = {"tr": stanza_pipeline("tr"), "kk": stanza_pipeline("kk"), "ky": stanza_pipeline("ky")}
tests = {k: read(k) for k in ["UD_Turkish-TueCL/tr_tuecl-ud-test.conllu", "UD_Azerbaijani-TueCL/az_tuecl-ud-test.conllu",
                              "UD_Kyrgyz-TueCL/ky_tuecl-ud-test.conllu", "UD_Kazakh-KTB/kk_ktb-ud-test.conllu"]}
TR, AZ, KY, KK = tests.values()

runs = [("Turkish TueCL", TR, "tr", None, "Turkish model, in-language reference"),
        ("Azerbaijani TueCL", AZ, "tr", None, "Turkish model, zero-shot (same script)"),
        ("Kyrgyz TueCL", KY, "tr", None, "Turkish model, Cyrillic input"),
        ("Kyrgyz TueCL", KY, "tr", ("kir", "CTS"), "Turkish model, CTS input"),
        ("Kyrgyz TueCL", KY, "ky", None, "Kyrgyz KTMU model (other treebank)"),
        ("Kazakh KTB", KK, "tr", None, "Turkish model, Cyrillic input"),
        ("Kazakh KTB", KK, "tr", ("kaz", "Latn"), "Turkish model, Kazakh Latin input"),
        ("Kazakh KTB", KK, "tr", ("kaz", "CTS"), "Turkish model, CTS input"),
        ("Kazakh KTB", KK, "kk", None, "Kazakh KTB model (replay diagnostic)")]
preds = {}
print(f"{'test set':18} {'condition':40} {'UPOS':>6} {'UAS':>6} {'LAS':>6}")
for test, gold, lang, conv, label in runs:
    forms = convert(gold, *conv) if conv else None
    pred = parse_gold_words(nlp[lang], gold, forms)       # predictions carry the original word forms
    preds[(test, label)] = pred
    r = score(gold, pred)
    print(f"{test:18} {label:40} {r['UPOS']:6.2f} {r['UAS']:6.2f} {r['LAS']:6.2f}")
print("example, Kyrgyz TueCL sentence 1:", [w.form for w in KY[0]["words"]], "->", convert(KY[:1], "kir", "CTS")[0])
print("lowest-recall relations, Turkish model on Kyrgyz TueCL (CTS):",
      per_relation(KY, preds[("Kyrgyz TueCL", "Turkish model, CTS input")]))
print("lowest-recall relations, Turkish model on Kazakh KTB (CTS):",
      per_relation(KK, preds[("Kazakh KTB", "Turkish model, CTS input")]))

## Worked solution

Output of `experiments/ch08_exercises/run_ch08_exercises.py` (UD 2.17; Stanza 1.11.0):

```
test set           condition                                  UPOS    UAS    LAS
Turkish TueCL      Turkish model, in-language reference      84.29  73.45  64.27
Azerbaijani TueCL  Turkish model, zero-shot (same script)    66.78  55.48  43.20
Kyrgyz TueCL       Turkish model, Cyrillic input             36.88  13.36   6.16
Kyrgyz TueCL       Turkish model, CTS input                  47.04  38.16  22.40
Kyrgyz TueCL       Kyrgyz KTMU model (other treebank)        68.40  46.80  29.12
Kazakh KTB         Turkish model, Cyrillic input             36.31  14.96   9.09
Kazakh KTB         Turkish model, Kazakh Latin input         47.81  29.31  18.07
Kazakh KTB         Turkish model, CTS input                  57.60  41.30  27.17
Kazakh KTB         Kazakh KTB model (replay diagnostic)      98.74  96.83  95.57
example, Kyrgyz TueCL sentence 1: ['Кыз', 'досуна', 'кат', 'жазды', '.'] -> ['Kız', 'dosuna', 'kat', 'cazdı', '.']
lowest-recall relations, Turkish model on Kyrgyz TueCL (CTS): aux 0% (n=78), ccomp 0% (n=22), discourse 3% (n=30), conj 4% (n=27), advcl 13% (n=38), obl 14% (n=104)
lowest-recall relations, Turkish model on Kazakh KTB (CTS): acl 0% (n=190), aux 0% (n=174), cc 0% (n=187), compound 0% (n=130), csubj 0% (n=73), dep 0% (n=108)
```

**Script is the first barrier.** On Cyrillic input the Turkish model is close to useless: LAS 6.16 on Kyrgyz TueCL and 9.09 on Kazakh KTB, with UPOS around 36%. Converting the words to CTS raises LAS to 22.40 (Kyrgyz) and 27.17 (Kazakh), a gain of 16–18 points. For Kazakh, the 2021 national Latin alphabet helps less (18.07) than CTS, which spells shared sounds the way Turkish does (*cazdı* rather than *jazdy*). The Turkish model on Azerbaijani, which needs no conversion, reaches LAS 43.20, against 64.27 on the Turkish versions of the same sentences.

**In-language models.** The Kyrgyz KTMU model reaches LAS 29.12 on Kyrgyz TueCL, only 6.7 points above the transliterated Turkish model. The TueCL sentences are constructed grammar examples, unlike the KTMU training text, and the two Kyrgyz treebanks may also differ in annotation; this comparison cannot separate the two. The Kazakh KTB model's 95.57 LAS is a replay of data it has seen and must not be compared with the other rows.

**Error-prone relations.** With CTS input, the Turkish model never gets `aux` or `ccomp` right on Kyrgyz TueCL, and gets 3–14% of `discourse`, `conj`, `advcl` and `obl` arcs. On KTB it never gets `acl`, `aux`, `cc`, `compound`, `csubj` or `dep` right. Some of these are convention differences rather than parsing failures: `dep` occurs once in the IMST training split but 108 times in the KTB test set, and the light-verb direction is reversed between IMST and KTB (Syntax and Universal Dependencies, light-verb table). Auxiliaries, conjunctions and other function words, which differ most between the languages lexically, are the other main source of errors.

Running the multilingual Glot500 parser on the Cyrillic input is the first "Your turn" task.

## Your turn — extension

1. Run the same table with `turkicnlp.Pipeline(..., pos_backend="multilingual_glot500_model", depparse_backend="multilingual_glot500_model")` on the Cyrillic input.
2. Score only the arcs whose gold relation is one of `nsubj`, `obj`, `obl`, `amod`, `nmod`. How much of the zero-shot gap remains on these core relations?